In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [2]:
INPUT_CSV  = "usa_europe_geotagged.csv"
OUT_DIR    = "context_embeddings"
CHUNKSIZE  = 10_000

WEATHER_MAP = {"sunny": 0, "cloudy": 1, "rainy": 2, "snowy": 3, "unknown": 4}
MOOD_MAP    = {"excited": 0, "happy": 1, "neutral": 2, "tired": 3, "unknown": 4}

COLUMNS = ["photoid", "uid_int", "hour", "month",
           "dow", "time_bucket", "weather_code", "mood_code"]

os.makedirs(OUT_DIR, exist_ok=True)
print("Config ready. Output dir:", OUT_DIR)

Config ready. Output dir: context_embeddings


In [3]:
df_head = pd.read_csv(INPUT_CSV, nrows=5)
print("Columns:", df_head.columns.tolist())
print()
df_head[['photoid', 'uid', 'datetaken', 'latitude', 'longitude', 'usertags']]

Columns: ['photoid', 'uid', 'unickname', 'datetaken', 'dateuploaded', 'capturedevice', 'title', 'description', 'usertags', 'machinetags', 'longitude', 'latitude', 'accuracy', 'pageurl', 'downloadurl', 'licensename', 'licenseurl', 'serverid', 'farmid', 'secret', 'secretoriginal', 'ext', 'marker']



,photoid,uid,datetaken,latitude,longitude,usertags
0,29872,34427465504@N01,2004-05-05 21:30:05.0,38.760236,-93.440796,"2004,bruise,me,photo,unfound"
1,29873,34427465504@N01,2004-05-05 21:30:06.0,38.760236,-93.440796,"2004,bruise,me,photo,unfound"
2,29874,34427465504@N01,2004-05-05 21:30:07.0,38.760236,-93.440796,"2004,bruise,me,photo,unfound"
3,29875,34427465504@N01,2004-05-05 21:30:09.0,38.760236,-93.440796,"2004,bruise,me,photo,unfound"
4,30431,34427469121@N01,2004-03-05 15:03:52.0,47.619673,-122.351903,"elevator,seattle,space+needle,space+travel"


In [4]:
def encode_time(df):
    dt  = pd.to_datetime(df['datetaken'], errors='coerce')

    hour  = dt.dt.hour.fillna(-1).astype(np.int8)
    month = dt.dt.month.fillna(-1).astype(np.int8)
    dow   = dt.dt.dayofweek.fillna(-1).astype(np.int8)  # 0=Monday, 6=Sunday

    h      = hour.values
    bucket = np.select(
        [h < 0,  h < 6,  h < 12,  h < 18],
        [4,      0,      1,       2     ],
        default=3
    ).astype(np.int8)

    return hour.values, month.values, dow.values, bucket


In [5]:
def encode_weather(month_arr, lat_arr):
    m   = np.asarray(month_arr, dtype=np.int8)
    lat = np.asarray(lat_arr)

    northern = lat >= 0
    out      = np.full(len(m), WEATHER_MAP['unknown'], dtype=np.int8)

    for is_north in [True, False]:
        idx = np.where((northern == is_north) & (m > 0))[0]
        mo  = m[idx]

        if is_north:
            w = np.select(
                [np.isin(mo, [12,1,2]), np.isin(mo, [3,4,5]), np.isin(mo, [6,7,8])],
                [WEATHER_MAP['snowy'],  WEATHER_MAP['cloudy'], WEATHER_MAP['sunny']],
                default=WEATHER_MAP['rainy']
            )
        else:
            w = np.select(
                [np.isin(mo, [12,1,2]), np.isin(mo, [3,4,5]), np.isin(mo, [6,7,8])],
                [WEATHER_MAP['sunny'],  WEATHER_MAP['rainy'],  WEATHER_MAP['snowy']],
                default=WEATHER_MAP['cloudy']
            )

        out[idx] = w.astype(np.int8)

    return out

In [6]:
POSITIVE_TAGS = frozenset({
    'happy', 'joy', 'fun', 'love', 'beautiful', 'amazing',
    'excited', 'celebration', 'party', 'vacation', 'holiday',
    'smile', 'beach', 'travel', 'adventure', 'awesome'
})

TIRED_TAGS = frozenset({
    'tired', 'exhausted', 'sleep', 'rest', 'night', 'dark',
    'cold', 'winter', 'grey', 'gray', 'fog', 'rain', 'cloudy'
})

def _single_mood(tags, tb):
    if not isinstance(tags, str) or tags.strip() == '':
        if tb in (1, 2): return MOOD_MAP['happy']
        if tb == 0:      return MOOD_MAP['tired']
        return MOOD_MAP['neutral']

    tag_set = frozenset(t.strip().lower() for t in tags.split(','))
    pos     = len(tag_set & POSITIVE_TAGS)
    neg     = len(tag_set & TIRED_TAGS)

    if pos > neg: return MOOD_MAP['excited'] if pos >= 3 else MOOD_MAP['happy']
    if neg > pos: return MOOD_MAP['tired']
    return MOOD_MAP['neutral']

def encode_mood(tags_series, bucket_arr):
    with ThreadPoolExecutor(max_workers=32) as ex:
        result = list(ex.map(_single_mood, tags_series, bucket_arr))
    return np.array(result, dtype=np.int8)

In [7]:
print("Pass 1: Collecting unique UIDs...")

uid_set = set()

for chunk in tqdm(pd.read_csv(INPUT_CSV, usecols=['uid'], chunksize=CHUNKSIZE)):
    uid_set.update(chunk['uid'].dropna().astype(str).values)

uid_list   = sorted(uid_set)
uid_to_int = {u: i for i, u in enumerate(uid_list)}

np.save("uid_keys.npy",   np.array(uid_list, dtype=object))
np.save("uid_values.npy", np.array(range(len(uid_list)), dtype=np.int32))

print(f"Unique users : {len(uid_list):,}")
print(f"Saved → uid_keys.npy + uid_values.npy")

Pass 1: Collecting unique UIDs...


768it [00:26, 28.45it/s]


Unique users : 91,779
Saved → uid_keys.npy + uid_values.npy


In [8]:
COLS = ['photoid', 'uid', 'latitude', 'longitude', 'datetaken', 'usertags']

print("Pass 2: Encoding & saving context chunks...")

total_rows   = sum(1 for _ in open(INPUT_CSV)) - 1  # count rows first
total_chunks = (total_rows // CHUNKSIZE) + 1

reader = pd.read_csv(INPUT_CSV, usecols=COLS, chunksize=CHUNKSIZE)

with tqdm(total=total_chunks, desc="Chunks", unit="chunk") as pbar:

    for i, chunk in enumerate(reader):

        chunk = chunk.dropna(subset=['latitude', 'longitude', 'datetaken']).reset_index(drop=True)
        if chunk.empty:
            pbar.update(1)
            continue

        hour, month, dow, bucket = encode_time(chunk)
        weather                  = encode_weather(month, chunk['latitude'].values)
        mood                     = encode_mood(chunk['usertags'].values, bucket)

        uid_int = np.array(
            [uid_to_int.get(str(u), -1) for u in chunk['uid'].values],
            dtype=np.int32
        )

        matrix = np.column_stack([
            chunk['photoid'].values.astype(np.int64),
            uid_int,
            hour, month, dow, bucket, weather, mood
        ]).astype(np.float32)

        np.save(os.path.join(OUT_DIR, f"context_chunk_{i}.npy"), matrix)

        pbar.set_postfix({"chunk": i, "rows": matrix.shape[0]})
        pbar.update(1)

print(f"\nDone! {i+1} chunks saved to '{OUT_DIR}/'")
print(f"Columns: {COLUMNS}")

Pass 2: Encoding & saving context chunks...


Chunks: 100%|██████████| 768/768 [10:10<00:00,  1.26chunk/s, chunk=767, rows=2417]


Done! 768 chunks saved to 'context_embeddings/'
Columns: ['photoid', 'uid_int', 'hour', 'month', 'dow', 'time_bucket', 'weather_code', 'mood_code']


In [9]:
sample = np.load(os.path.join(OUT_DIR, "context_chunk_0.npy"))

print("Shape        :", sample.shape)
print("Columns      :", COLUMNS)
print("NaN count    :", np.isnan(sample).sum())
print("Sample row   :", sample[0])

Shape        : (10000, 8)
Columns      : ['photoid', 'uid_int', 'hour', 'month', 'dow', 'time_bucket', 'weather_code', 'mood_code']
NaN count    : 0
Sample row   : [2.9872e+04 3.1070e+04 2.1000e+01 5.0000e+00 2.0000e+00 3.0000e+00
 1.0000e+00 2.0000e+00]


In [ ]:
ids_dir = "ids"

id_files    = sorted([f for f in os.listdir(ids_dir) if f.endswith('.npy')])
visual_ids  = set(np.concatenate([np.load(os.path.join(ids_dir, f)) for f in id_files]).astype(int).tolist())

ctx_files   = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('.npy')])
ctx_ids     = set(np.concatenate([
    np.load(os.path.join(OUT_DIR, f))[:, 0].astype(int) for f in ctx_files
]).tolist())

overlap = visual_ids & ctx_ids

print(f"Visual embedding IDs : {len(visual_ids):>10,}")
print(f"Context feature IDs  : {len(ctx_ids):>10,}")
print(f"Overlap              : {len(overlap):>10,}")
print(f"Coverage             : {len(overlap)/len(visual_ids)*100:.1f}%")